# 远程因子库服务

远程服务器配置多个真实因子库，客户端通过 `RemoteFactorDB → RemoteFactorTable → RemoteFactor` 访问数据。数据源读取与写入在服务端执行；衍生因子、算子、计算图和回测始终在本地执行。

服务独立于 QSWeb。第一版已对照验证 DuckDB 原生表、DuckDB Parquet 视图和 HDF5DB。其他后端需要在目标环境单独验证，重写 `readData()` 或返回特殊原子因子类的后端需要显式适配器。

详细架构和失败语义见 [设计方案](../plans/远程因子库服务设计方案.md)。

## 安装与服务启动

服务端在 QSExt 仓库执行 `python -m pip install -e '.[factor-server]'`，并安装实际后端依赖；客户端执行 `python -m pip install -e '.[factor-client]'`。

复制 `examples/FactorService/settings.py` 到部署目录，配置 `FACTOR_DATABASES` 与 `FACTOR_SERVICE`。`class` 支持已有内置短名或完整类路径，DuckDB 请使用完整类路径。

通过环境变量 `QS_FACTOR_DATA_ROOT` 指定数据根目录，`QS_FACTOR_SERVICE_TOKEN` 提供服务令牌。令牌必须设置且不应写入配置文件或版本库。HDF5DB 的目录需存在，或将业务角色设置为 `target`/`proxy` 以沿用已有工厂的建目录行为。

从 QSExt 根目录运行：

```powershell
python scripts/run_factor_service.py --settings path/to/settings.py
```

默认监听本机地址。远程部署时配置监听地址并由反向代理提供 HTTPS。仅使用一个 HTTP worker，不启用热重载或多 worker。

`FACTOR_SERVICE.Databases` 是显式导出清单；`Access='read'` 或 `'read_write'` 控制权限，与 `role` 无关。每个导出项的 `ResourceID` 必须唯一，指向不同后端文件的库不应复用同一个资源身份。相同文件的多别名配置首版会被拒绝。

In [ ]:
import os
import datetime as dt

from QSExt.Factor.RemoteDB import FactorDBServiceClient, RemoteFactorDB

# 在本机环境设置 URL 和客户端令牌，令牌值与服务器一致。
server_url = os.environ['QS_FACTOR_SERVER_URL']
token_env = 'QS_FACTOR_CLIENT_TOKEN'

## 发现与选择因子库

目录仅包含公开标识、访问模式和连接状态，不返回服务器路径或数据库凭据。目录查询不强制连接每个后端。以下示例选择配置中的 `market`，表和字段需要替换成目标库的实际名称。

In [6]:
with FactorDBServiceClient(server_url, token_env=token_env) as service:
    databases = service.listDatabases()
    print(databases)

fdb = RemoteFactorDB(args={
    'ServerURL': server_url,
    'Database': 'LDB',
    'TokenEnv': token_env,
    'TimeoutSeconds': 120,
}).connect()

print(fdb.TableNames)
print(fdb.Capabilities)

[{'name': 'JYDB', 'resource_id': 'qsweb-jydb', 'config_revision': '5bf9aaaeb2b62b756c7c818194ea0450ff66a88a5f9b017fe93c3a4f61361e89', 'access': 'read', 'status': 'not_connected'}, {'name': 'BSDB', 'resource_id': 'qsweb-bsdb', 'config_revision': 'd20cf640a238688718f563f0433c25ee70849f11373eddc6acfdcb2911083a9c', 'access': 'read', 'status': 'not_connected'}, {'name': 'ASDB', 'resource_id': 'qsweb-asdb', 'config_revision': 'fdcf973f55b165bd808619a9197ffd03333ab0c1479a2ab17c28fb0c43400d78', 'access': 'read', 'status': 'not_connected'}, {'name': 'LDB', 'resource_id': 'qsweb-ldb', 'config_revision': '5998666d80c5da87192240a9e40c363fc149d3734d21a6286b3afe6ee4085e92', 'access': 'read', 'status': 'not_connected'}]
['index_cn_asset_alloc_strategy', 'index_cn_day_bar', 'index_cn_factor_alternative_from_constituent', 'index_cn_factor_growth_from_constituent', 'index_cn_factor_liquidity_from_constituent', 'index_cn_factor_momentum_from_constituent', 'index_cn_factor_quality_from_constituent', 'inde

## 表、因子及数据读取

读取结果分别为项目原有的 Panel 与 pandas DataFrame。轴顺序、dtype、全缺失位置和空结果通过标准数据封装恢复；不自动裁剪证券、日期或因子。`None` 与空列表原样交给后端处理，后端不支持的输入会报错。

In [7]:
table = fdb.getTable('index_cn_day_bar', args={'LookBack': 0})
print(table.FactorNames)
print(table.getMetaData())
print(table.getFactorMetaData())

ids = table.getID()
dates = table.getDateTime(
    start_dt=dt.datetime(2026, 1, 1),
    end_dt=dt.datetime(2026, 1, 31),
)
values = table.readData(factor_names=['close', 'open'], ids=ids, dts=dates)
close = table.getFactor('close')
close_values = close.readData(ids=ids, dts=dates)

['amount', 'chg_rate', 'close', 'float_cap', 'high', 'low', 'open', 'pre_close', 'volume']
Author                                                          麦冬
DefScriptPath    D:\Project\QSExt\QSExt\FactorDef\JY\index_cn_d...
Description                                   指数的行情数据, 包括开高低收、成交量等
IDType                                                          指数
dtype: object
          DataType                                     SourceFactorID
amount      double  a7acd4d1e703e70689dddb90f792661f241a5ce0e92e4e...
chg_rate    double  0e7b33bce5ef4d7053ff5b30497f5660f8899d64e81b08...
close       double  ba04b49e12b32b4e94d5e077a1558809b1fda53e0b1b5c...
float_cap   double  016734c451d5c886b886f6ed17c29a69e00c9cea1f7a6a...
high        double  7ca61019802e70089934811e589bbee2cc2f4018068304...
low         double  c0a9bdbea15a315965fe2a4905af9670b3e8f216aadd93...
open        double  6aa289e7f3c3544dce5a1568abe6a0df7da361da8f32b0...
pre_close   double  4335ef0eb4a404950fe6c0f2e3c1a15911ff97edf4d849.

## 本地衍生计算与缓存

远程原子因子仍是本地 Factor，算术或框架算子会构成本地依赖图。服务端只接收白名单数据源操作，不接收算子或函数。

代理覆盖了标准因子缓存路径：本地准备阶段合并兼容读取为 HTTP 批次，在本地保存标准数据；不传输各后端私有的原始数据结构。时间窗口使用本地引擎展开后的日期，截面和多进程分片保留本地框架约定。

In [11]:
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache

open_ = table.getFactor('open')
returns = close / open_ - 1

with FeatherFactorCache(args={'DTRuler': dates}) as cache:
    with FactorContext(DTRuler=dates, SectionIDs=ids, DataCache=cache):
        result = returns.readData(ids=ids, dts=dates)

### 数据版本与多进程

服务第一版不提供后端可信数据快照，读取时间不是数据版本。未传 `DataVersion` 时，框架使用运行身份隔离缓存；只有调用者能够确保源数据不可变时，才应显式设置可跨运行复用的 `DataVersion`。

不要在一次读取或本地运行中假定跨请求事务一致性，也不要在源数据变化后继续复用相同的可信版本标签。

Windows spawn 和 ParallelEngine 已有真实进程对照测试。HTTP 连接、socket 和锁不参与代理序列化，子进程从自己的环境重新读取令牌并创建连接。工作进程必须具有同名认证环境变量。

## 参数与对象重建

后端参数通过 `getTable(args=...)`、`getFactor(args=...)` 和 `.new(args=...)` 传递，在服务端校验。代理中的后端表参数位于 `table.Args.BackendArgs`；其布局与原生后端的扁平 Args 不同。

使用 `.new()` 创建新对象，不原地修改嵌套参数字典。代理会拒绝已被篡改的读取描述。不同参数的同名因子保持不同的语义身份及缓存。

In [13]:
lookback_table = table.new(args={'LookBack': 5})
print(lookback_table.Args.BackendArgs)

uncached_close = close.new(args={'CacheEnabled': False})
print(uncached_close.Args.CacheEnabled)

{'LookBack': 5, 'OnlyStartLookBack': False, 'OnlyLookBackNontarget': False, 'OnlyLookBackDT': False, 'TargetDT': None}
False


## 写入与元信息维护

真实后端支持、服务白名单允许、且服务端访问模式为 `read_write` 的操作才能执行。公开写入方法保留原生后端的语义；不将顺序执行视为额外事务保证。DuckDB 的 Parquet 视图不支持数据写入。

以下代码会修改所选远程库，运行前应确认目标库和表。

In [ ]:
import pandas as pd
from QuantStudio.Core.QSObject import Panel

writer = RemoteFactorDB(args={
    'ServerURL': server_url,
    'Database': 'research',
    'TokenEnv': token_env,
}).connect()

example = pd.DataFrame([[1.0]], index=[dt.datetime(2026, 1, 1)], columns=['EXAMPLE'])
writer.writeData(Panel({'value': example}), 'remote_example', if_exists='update')
writer.setTableMetaData('remote_example', key='Description', value='远程写入示例')
print(writer.getOperation(writer.LastOperationID))

### 写入超时

每次写入只提交一次，不自动重试。超时或网络断开不表示服务器已经回滚；异常的 `operation_id` 或库的 `LastOperationID` 可用于状态查询。

同一服务进程、有效期内的相同操作标识会去重。后端失败可能部分生效，服务重启后的旧写入结果可能为 `UNKNOWN`。第一版没有持久化操作日志，也不承诺跨重启恰好一次。未知状态必须核对数据后再处理。

In [ ]:
from QSExt.FactorService.protocol import RemoteFactorError

try:
    writer.setFactorMetaData('remote_example', 'value', key='Description', value='示例因子')
except RemoteFactorError as error:
    operation_id = error.operation_id or writer.LastOperationID
    if operation_id:
        print(writer.getOperation(operation_id))
    raise

## 既有 RuntimeConfig 与 QSWeb 配置

`FACTOR_DATABASES` 中使用完整代理类路径即可由 DBPool 加载：

```python
FACTOR_DATABASES = [{
    'name': 'remote_market',
    'class': 'QSExt.Factor.RemoteDB.RemoteFactorDB',
    'role': 'source',
    'args': {
        'ServerURL': 'https://factor.example.com',
        'Database': 'market',
        'TokenEnv': 'QS_FACTOR_CLIENT_TOKEN',
    },
}]
```

因子定义仍通过 `fdi.FDB['remote_market'].getTable(...)` 使用。QSWeb 的连接服务和因子服务已增加该代理类型的工厂分支。QSWeb 原有预览接口的裁剪行为仍属于其 UI 预览逻辑，不改变此服务的完整数据读取契约。

## 支持范围与限制

- 标准公开读接口、原生可写接口，以及 DuckDB 的建表、增加因子、删除数据和删除元信息覆盖等已注册扩展。
- 扩展可通过 `callExtension()` 调用；连接后也可按已发现的方法名调用。未知方法不会被任意反射执行。
- SQL 文本、转义模板、日期格式等参数使用服务器预配置，不接受客户端覆盖；筛选支持字段比较、算术、位逻辑及 isin/isna/notna，不执行 Python eval。尚未登记的后端扩展查询需验证后再开放。
- 元信息与标准数据支持数值、布尔、字符串、日期时间、受限列表/元组/字典及缺失值。自定义对象、函数和分类 dtype 等未支持类型明确拒绝；不使用 pickle/dill 作为网络数据格式。
- 协议有请求、结果、单元格数和队列上限。首版使用有界完整响应，尚未提供大结果异步下载或 Arrow 流式 HTTP 输出；超限会报错，不静默截断。
- 文件型数据库应由本服务持有访问权，外部进程的更新仍需遵循后端并发限制。
- 参数与编码错误、权限错误及资源变化使用 RemoteFactorError，包含结构化错误码。服务器详细日志按请求标识关联。
- 不提供任意 SQL、连接或游标代理、客户端代码上传、远程事务和远程计算图。

In [ ]:
writer.disconnect()
fdb.disconnect()

## 同机 QSWeb 与独立 FactorService 共享 DuckDB

在共享 settings 的原生 DuckDB 定义中指定磁盘 `DBFile`，并设置 `CoordinatedAccess=True, LockTimeout=30`。QSWeb 继续直接访问本地 DuckDB，FactorService 则为远程客户端代理同一文件；计算图保留在各自调用端。

读写在文件锁内使用短连接和完整事务，同一文件的访问串行。所有访问进程必须同时启用该模式，切换前关闭旧长连接；原生数据库工具不参与协调。该模式仅用于同机本地磁盘，不为 SMB/NFS 跨主机访问提供保证。它不协调外部 Parquet 写入，也不保证跨请求或已有缓存使用同一数据版本。

`:memory:` 不支持协调模式；独立内存库读取 Parquet 无需该锁，共享磁盘元信息库需明确指定 DBFile，或使用目录内已有的 `_metadata.duckdb`。完整会话用法见 [DuckDB](DuckDB.ipynb)，共享 settings 示例见 `examples/FactorService/settings.py`。
